In [2]:
import pandas as pd
import numpy as np
import json

rural_df = pd.read_pickle('../data/interim/rural_ir.pkl')

with open('../data/interim/variable_labels.json') as f:
    variable_labels = json.load(f)

print("Shape:", rural_df.shape)

Shape: (6826, 5968)


In [3]:
final_features = {
    'target_anemia': 'v457',
    'wealth': 'v190a',
    'education': 'v106',
    'education_years': 'v133',
    'antenatal_visits': 'm14_1',
    'toilet': 'v116',
    'water': 'v113',
    'age': 'v012',
    'age_group': 'v013',
    'province': 'v024',
    'ecological_region': 'secoreg',
    'altitude': 'v040',
    # --- new causal-pathway features ---
    'bmi': 'v445',
    'currently_pregnant': 'v213',
    'children_ever_born': 'v201',
    'births_last_5yr': 'v208',
    'contraceptive_method': 'v312',
    'currently_breastfeeding': 'v404',
    'smokes': 'v463a',
}

cols = list(final_features.values())
df = rural_df[cols].copy()
df.shape

(6826, 19)

In [4]:
# m14_1: code 98 = "don't know" — treat as missing, not a real visit count
df['m14_1'] = df['m14_1'].replace(98, np.nan)

# v116/v113 had rare codes like 96, 97 in your earlier output — these are typically
# "other"/"missing" placeholders in DHS. Confirm against your codebook; flagged here for now.
print(df['v116'].value_counts())
print(df['v113'].value_counts())

v116
13    3484
12    1835
31     499
97     373
45     284
22     226
21      61
23      34
11      14
14      11
15       4
96       1
Name: count, dtype: int64
v113
12    2842
21    1994
14    1040
97     373
11     126
42     123
13     111
41     107
32      59
31      19
43      17
71      14
96       1
Name: count, dtype: int64


In [5]:
# v457: 1=severe, 2=moderate, 3=mild anemia, 4=not anemic (confirmed convention — verify against your recode manual)
df['anemia_binary'] = df['v457'].map({1: 1, 2: 1, 3: 1, 4: 0})

# Drop rows with no hemoglobin test (can't have ground truth without it)
before = df.shape[0]
df = df[df['anemia_binary'].notna()].copy()
after = df.shape[0]

print(f"Dropped {before - after} untested women ({(before-after)/before*100:.1f}%)")
print(f"Final modeling sample: {after}")
print(df['anemia_binary'].value_counts(normalize=True) * 100)

Dropped 3455 untested women (50.6%)
Final modeling sample: 3371
anemia_binary
0.0    68.436666
1.0    31.563334
Name: proportion, dtype: float64


In [6]:
# Drop rows with no hemoglobin test (can't have ground truth without it)
before = df.shape[0]
df = df[df['anemia_binary'].notna()].copy()
after = df.shape[0]

print(f"Dropped {before - after} untested women ({(before-after)/before*100:.1f}%)")
print(f"Final modeling sample: {after}")
print(df['anemia_binary'].value_counts(normalize=True) * 100)

Dropped 0 untested women (0.0%)
Final modeling sample: 3371
anemia_binary
0.0    68.436666
1.0    31.563334
Name: proportion, dtype: float64


In [7]:
df.isnull().mean().sort_values(ascending=False) * 100

m14_1            79.05666
v457              0.00000
v040              0.00000
v463a             0.00000
v404              0.00000
v312              0.00000
v208              0.00000
v201              0.00000
v213              0.00000
v445              0.00000
secoreg           0.00000
v190a             0.00000
v024              0.00000
v013              0.00000
v012              0.00000
v113              0.00000
v116              0.00000
v133              0.00000
v106              0.00000
anemia_binary     0.00000
dtype: float64

In [8]:
# --- m14_1: create flag + fill ---
df['had_recent_birth'] = df['m14_1'].notna().astype(int)
df['antenatal_visits_filled'] = df['m14_1'].fillna(0)

print(df['had_recent_birth'].value_counts(normalize=True) * 100)

had_recent_birth
0    79.05666
1    20.94334
Name: proportion, dtype: float64


In [9]:
# --- drop redundant columns ---
df = df.drop(columns=['v457', 'm14_1'])
print(df.shape)
print(df.isnull().sum())  # should be all zeros now

(3371, 20)
v190a                      0
v106                       0
v133                       0
v116                       0
v113                       0
v012                       0
v013                       0
v024                       0
secoreg                    0
v040                       0
v445                       0
v213                       0
v201                       0
v208                       0
v312                       0
v404                       0
v463a                      0
anemia_binary              0
had_recent_birth           0
antenatal_visits_filled    0
dtype: int64


In [10]:
# Save cleaned dataset — no encoding/scaling yet, that's feature engineering's job
df.to_pickle('../data/interim/rural_cleaned.pkl')
print("Saved cleaned dataset:", df.shape)
df.dtypes

Saved cleaned dataset: (3371, 20)


v190a                         int8
v106                          int8
v133                          int8
v116                          int8
v113                          int8
v012                          int8
v013                          int8
v024                          int8
secoreg                       int8
v040                         int16
v445                       float64
v213                          int8
v201                          int8
v208                          int8
v312                          int8
v404                          int8
v463a                         int8
anemia_binary              float64
had_recent_birth             int64
antenatal_visits_filled    float64
dtype: object